In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import boxcox
from scipy.special import inv_boxcox
from tqdm import tqdm
import re
import joblib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchsummary import summary

from warnings import filterwarnings
filterwarnings('ignore')

sns.set_style('darkgrid')
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Загрузка датасета

- Algo - 0
- Analysis - 1
- NN - 2
- Optim - 3
- SQL - 4

In [3]:
import h5py

In [4]:
with h5py.File('/kaggle/input/5-classes-clean/embeddings_dataset.h5', 'r') as f:
    embeddings = np.squeeze(f['embeddings'][:])
    labels = f['labels'][:]

In [5]:
embeddings.shape

(21254, 768)

In [6]:
labels.shape

(21254,)

# Классический МЛ

In [16]:
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, classification_report, accuracy_score, PrecisionRecallDisplay, RocCurveDisplay

In [9]:
X_train, X_test, y_train, y_test = train_test_split(embeddings, labels, test_size=0.1)

In [10]:
pd.Series(y_train).value_counts()

4    6616
1    4127
2    3566
0    3340
3    1479
Name: count, dtype: int64

In [62]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [65]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Полученное accuracy: {accuracy:.4f}")

Полученное accuracy: 0.8024


## Xgboost, lgbm, catboost с донастройкой на кросс-валидации

In [17]:
cv = StratifiedKFold(n_splits=5, shuffle=True)

In [18]:
def train_xgboost(X, y):
    params = {
        'objective': 'multi:softmax',
        'num_classes': len(np.unique(y)),
        'learning_rate': 0.1,
        'max_depth': 8,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'n_estimators': 1000,
        'eval_metric': 'mlogloss',
        'tree_method': 'gpu_hist',
        'predictor': 'gpu_predictor',
        'gpu_id': 0
    }

    model = XGBClassifier(**params)

    cv_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    print(f'Базовый XGBoost на кросс-валидации: {cv_scores.mean():.4f} +- {cv_scores.std() * 2:.4f}')

    return cv_scores